In [ ]:
import subprocess, sys, os

# Install from offline packages (internet disabled during Save Version)
offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
if os.path.exists(offline_dir):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                          "--no-index", "--find-links", offline_dir, 
                          "datasets", "trl"])
    print("Installed from offline packages")
else:
    # Fallback: try online
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "trl"])
    print("Installed from PyPI")

import datasets, trl
print("datasets:", datasets.__version__, "trl:", trl.__version__)

In [ ]:
import os
import sys
import stat
import shutil
import gc
import zipfile

import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# === Triton fixes ===
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

# === Hyperparameters ===
SUBSAMPLE_SIZE = 600
LORA_RANK = 32
LORA_ALPHA = 64          # 2x rank (QLoRA paper recommendation)
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 2e-4
NEFTUNE_ALPHA = 5.0      # NEFTune noise for generalization
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Data ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Total training samples: {len(train_df)}")
train_df = train_df.sample(n=SUBSAMPLE_SIZE, seed=42)

hf_dataset = Dataset.from_pandas(train_df.to_pandas())

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    prompt = example["prompt"]
    answer = example["answer"]
    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    assistant_msg = f"\\boxed{{{answer}}}"
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)
print(f"Dataset formatted. Example:\n{hf_dataset[0]['text'][:500]}")

# === Model ===
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# === Triton compiler fix ===
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# === Training ===
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    neftune_noise_alpha=NEFTUNE_ALPHA,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
import zipfile, os

OUTPUT_DIR = "/kaggle/working/adapter"
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
with zipfile.ZipFile(zip_path, 'r') as zf:
    print(f"Contents: {zf.namelist()}")
    assert "adapter_config.json" in zf.namelist()
print("submission.zip ready!")

In [ ]:
print('Done.')